In [ ]:
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)


In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

analytics_mongo = client_db.get_database('analytics_db')


In [ ]:
movement_similarity_vehicles_latest=pd.DataFrame(list(analytics_mongo.vehicle_cluster_priority.find()))
#pd.DataFrame("vehicle_cluster_priority.csv")

In [ ]:
movement_similarity_vehicles_latest.vehicle_number.nunique()

In [ ]:
clusters=pd.DataFrame(list((analytics_mongo.location_clusters.find({}))))

In [ ]:
clusters.to_csv("location_clusters.csv")

In [ ]:
clusters=pd.read_csv("location_clusters.csv")

In [ ]:
movement_similarity_vehicles_latest.fillna(-1,inplace=True)

In [ ]:
clusters=clusters[['clusters','state','region']]

In [ ]:
movement_similarity_vehicles_latest=movement_similarity_vehicles_latest[['vehicle_number','cluster1', 'priority1', 'cluster2', 'priority2', 'cluster3', 'priority3', 'cluster4', 'priority4', 'cluster5', 'priority5', 'cluster6', 'priority6', 'cluster7', 'priority7', 'cluster8', 'priority8', 'cluster9', 'priority9', 'cluster10', 'priority10']]

In [ ]:
movement_similarity_vehicles_latest=pd.merge(movement_similarity_vehicles_latest,clusters,left_on='cluster1',right_on='clusters',how='left')

In [ ]:
del movement_similarity_vehicles_latest['clusters']

In [ ]:
movement_similarity_vehicles_latest.rename(columns={'state':'state1','region':'region1'},inplace=True)

In [ ]:
for i in range(2,10,1):
    movement_similarity_vehicles_latest=pd.merge(movement_similarity_vehicles_latest,clusters,left_on='cluster{}'.format(i),right_on='clusters',how='left')
    del movement_similarity_vehicles_latest['clusters']
    movement_similarity_vehicles_latest.rename(columns={'state':'state{}'.format(i),'region':'region{}'.format(i)},inplace=True)
    
    
    
    
    

In [ ]:
regions=[]
for i, row in movement_similarity_vehicles_latest.iterrows():
    #print("*********")
    SOUTH=0
    NORTH=0
    WEST=0
    EAST=0
    NORTH_EAST=0
    CENTRAL=0
    
    
    for i in range(1,10,1):
        if row['cluster{}'.format(i)]>-1:
            if row['region{}'.format(i)]=='North':
                NORTH+=row['priority{}'.format(i)]
            if row['region{}'.format(i)]=='South':
                SOUTH+=row['priority{}'.format(i)]
            if row['region{}'.format(i)]=='East':
                EAST+=row['priority{}'.format(i)]
            if row['region{}'.format(i)]=='West':
                WEST+=row['priority{}'.format(i)]
            if row['region{}'.format(i)]=='Central':
                CENTRAL+=row['priority{}'.format(i)]
            if row['region{}'.format(i)]=='North East':
                NORTH_EAST+=row['priority{}'.format(i)]
                
        else :
            break 
    final_region=''
    
    if NORTH>20:
        final_region=final_region+' NORTH'
    if SOUTH>20:
        final_region=final_region+' SOUTH'
    if EAST>20:
        final_region=final_region+' EAST'
    if WEST>20:
        final_region=final_region+' WEST'
    if CENTRAL>20:
        final_region=final_region+' CENTRAL'
    if NORTH_EAST>20:
        final_region=final_region+' NORTH_EAST'
        
    regions.append(final_region)
    if (final_region)=='':
        print(row['vehicle_number'])
    
            
                
            

        
            

In [ ]:
movement_similarity_vehicles_latest['region_movement']=regions

In [ ]:
movement_similarity_vehicles_latest


In [ ]:
movement_similarity_vehicles_latest[['vehicle_number','region_movement']].to_csv("data/vehicle_movement_regions_NSWE_2020_Q1.csv")  


In [ ]:
movement_similarity_vehicles_latest.vehicle_number.nunique()